# 01 · Load the data

**Project:** _title_ · **Team:** _names_

Downloads the raw data once, checks it, and caches it as Parquet in `data/cache/` (git-ignored). Re-run it only when you want fresh data, and say so in the README.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv()                                   # tokens live in ../.env, never in the notebook

## Metadata

Name, publisher, licence and last update of each dataset (see `03_open_data_apis` in the course repository).

In [ ]:
DATASET = "xxxx-xxxx"   # Socrata id of your dataset
# meta = requests.get(f"https://analisi.transparenciacatalunya.cat/api/views/{DATASET}.json", timeout=30).json()
# meta["name"], meta.get("attribution"), meta.get("license")

## Download (all rows) and cache

In [ ]:
def cached(name, fetch):
    """Return data/cache/<name>.parquet; download it with fetch() the first time."""
    path = CACHE / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df


# df = cached("raw", lambda: fetch_socrata(DATASET))   # fetch_socrata: see 03_open_data_apis
# print(len(df), "rows")

## Checks

Row count against the server, types, missing values.

In [ ]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
    })


# profile(df)

Mearge two files

In [2]:
DATA1 = Path('../data/processed')
DATA2 = Path('../data/external')
#CACHE = DATA / 'cache'
domains_df = pd.read_parquet(DATA1 / 'all_publications_enriched.parquet')
researchers_df = pd.read_parquet(DATA2 / 'papers_cleaned.gencat.parquet')
# Merge on DOI
merged_df = researchers_df.merge(
    domains_df[["doi", "domain_name", "year"]],
    on="doi",
    how="left"
)

# Keep columns in the desired order
merged_df = merged_df[[
    "year",
    "doi",
    "orcidResearchers",
    "domain_name"
]]

# Save as parquet
output_file = "merged_data.parquet"
merged_df.to_parquet(output_file, index=False)

print(f"Saved {len(merged_df):,} rows to {output_file}")
print(merged_df.head())

KeyboardInterrupt: 

## Dataset card

| Field | Answer |
|---|---|
| Source (URL) and licence | |
| Rows × columns | |
| What is one row? | |
| Columns we use | |
| Missing values / problems | |
| API access? | |